# Compositing photos stored as bytes

A color photo is an array of shape `(H, W, 3)` with dtype `uint8`: one byte per channel,
a whole number from 0 to 255. As the lesson says, arithmetic on a `uint8` array, even with
a plain Python integer that fits in 0 to 255, stays `uint8` and wraps around modulo 256.
Here you paste one photo into another through a mask, brighten a photo, and find its
brightest pixel, each without a Python loop and without letting a byte wrap.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
fg = rng.integers(0, 256, size=(4, 5, 3), dtype=np.uint8)   # foreground photo
bg = rng.integers(0, 256, size=(4, 5, 3), dtype=np.uint8)   # background photo
mask = np.zeros((4, 5), dtype=bool)
mask[1:3, 1:4] = True                                       # where the foreground goes
print(fg.shape, fg.dtype, mask.sum(), "pixels in the mask")

## Part 1 (coding task): Paste through a mask

Write `composite(fg, bg, mask)`. `fg` and `bg` have shape `(H, W, 3)` and `mask` has shape
`(H, W)`. The result has the shape and dtype of `fg`, takes all three channels of a pixel
from `fg` where `mask` is true and from `bg` elsewhere, and leaves `fg` and `bg` unchanged.

In [ ]:
def composite(fg, bg, mask):
    """fg where mask is true, bg elsewhere, all three channels of each pixel."""
    # TODO
    raise NotImplementedError

In [ ]:
fg_before = fg.copy()
bg_before = bg.copy()
out = composite(fg, bg, mask)
assert np.array_equal(fg, fg_before), "composite changed fg itself: build a new array, or write into a copy of fg, never through fg"
assert np.array_equal(bg, bg_before), "composite changed bg itself: write into a copy, or build a new array"
assert out.shape == fg.shape, f"expected shape {fg.shape}, got {out.shape}"
assert out.dtype == np.uint8, f"expected dtype uint8, got {out.dtype}"
assert np.array_equal(out[mask], fg[mask]), "pixels inside the mask should come from fg, all three channels"
assert np.array_equal(out[~mask], bg[~mask]), "pixels outside the mask should come from bg, all three channels"
print("Part 1 passed")

## Part 2 (coding task): Brighten with a gain

Write `brighten(img, k)` for a whole number $k \ge 1$. It multiplies every channel of every
pixel by $k$, and a result above 255 stays at 255 (it saturates instead of wrapping). The
result has dtype `uint8`, and `img` itself is left unchanged.

In [ ]:
def brighten(img, k):
    """Every channel times k, capped at 255, as uint8."""
    # TODO
    raise NotImplementedError

In [ ]:
one = lambda v: np.full((1, 1, 3), v, dtype=np.uint8)
out = brighten(one(200), 2)
assert out.dtype == np.uint8, f"expected dtype uint8, got {out.dtype}"
assert out[0, 0, 0] != 144, "200 * 2 wrapped to 400 - 256 = 144 before the cap: multiply in a wider dtype"
assert out[0, 0, 0] == 255, f"200 * 2 = 400 should saturate at 255, got {out[0, 0, 0]}"
assert brighten(one(60), 3)[0, 0, 0] == 180, "60 * 3 = 180 is below the cap and should stay 180"
fg_before = fg.copy()
expected = np.minimum(fg.astype(int) * 3, 255).astype(np.uint8)
assert np.array_equal(brighten(fg, 3), expected), "wrong values on the foreground photo with k = 3"
assert np.array_equal(fg, fg_before), "brighten changed its input: build a new array instead of multiplying in place"
print("Part 2 passed")

## Part 3 (coding task): The brightest pixel by luma

Video and JPEG codecs rank how bright a pixel looks by its **luma**, a weighted sum of the
channels. With whole-number weights it is

$$
Y = \left\lfloor \frac{77R + 150G + 29B}{256} \right\rfloor ,
$$

a whole number from 0 to 255, since the weights add up to 256. Write `brightest(img)`,
which returns `(row, col)` as two Python integers: the pixel of an `(H, W, 3)` image with
the largest luma, the first in row order when several tie. Use `argmax` on the table of
lumas, which returns a position in the row order of the flattened table, and turn that
position into a row and a column yourself.

In [ ]:
def brightest(img):
    """(row, col) of the pixel with the largest luma, the first in row order on a tie."""
    # TODO
    raise NotImplementedError

In [ ]:
img = np.zeros((4, 5, 3), dtype=np.uint8)
img[2, 3] = [200, 220, 90]      # luma 199, the brightest
img[0, 1] = [0, 0, 35]          # luma 3, a dark blue
r = tuple(int(v) for v in brightest(img))
assert r not in [(0, 0), (0, 1)], "the products wrapped in uint8 (77 * 200 = 15400 does not fit in a byte), so the lumas are wrong: convert to a wider dtype before multiplying"
assert r != (3, 1), "position 13 was split by the number of rows: each row of the flattened table holds W = 5 pixels"
assert r == (2, 3), f"expected (2, 3), got {r}"
img[1, 0] = [200, 220, 90]      # a tie, earlier in row order
assert tuple(int(v) for v in brightest(img)) == (1, 0), "on a tie take the first pixel in row order"
row, col = brightest(fg)
luma = (fg.astype(int) @ np.array([77, 150, 29])) // 256
assert luma[row, col] == luma.max(), "the pixel returned for fg does not have the largest luma"
print("Part 3 passed")

## Solution

### Part 1: Paste through a mask

In [ ]:
def composite(fg, bg, mask):
    """fg where mask is true, bg elsewhere, all three channels of each pixel."""
    return np.where(mask[..., None], fg, bg)     # (H, W, 1) stretches over the channels

**Why it works.** `np.where` broadcasts its three arguments together. `mask` alone has
shape `(H, W)`, which lines up from the right against `(H, W, 3)` as $W$ against 3 and
fails. `mask[..., None]` has shape `(H, W, 1)`, so the 1 stretches over the three channels
and each pixel is taken whole. `np.where` builds a new array, so `bg` is left as it was. The
other common route, `out = bg` followed by `out[mask] = fg[mask]`, writes into `bg` itself,
because `out` is only a second name for the same array: start from `bg.copy()` instead.

### Part 2: Brighten with a gain

In [ ]:
def brighten(img, k):
    """Every channel times k, capped at 255, as uint8."""
    scaled = img.astype(np.int64) * k            # room for values above 255
    return np.minimum(scaled, 255).astype(np.uint8)

**Why it works.** `img * k` with `img` in `uint8` and `k` a Python integer that fits in a byte stays in `uint8`,
so $200 \cdot 2 = 400$ is stored as $400 - 256 = 144$, which is below 255, and the cap
never sees the overflow. `np.minimum(img * k, 255)` and `np.clip(img * k, 0, 255)` both
cap the wrapped value. Converting to `int64` first keeps 400, the cap brings it to 255, and
only then does the result go back to `uint8`, where every value now fits.

### Part 3: The brightest pixel by luma

In [ ]:
def brightest(img):
    """(row, col) of the pixel with the largest luma, the first in row order on a tie."""
    rgb = img.astype(np.int64)                   # 77 * 255 alone needs more than a byte
    luma = (77 * rgb[..., 0] + 150 * rgb[..., 1] + 29 * rgb[..., 2]) // 256
    flat = int(luma.argmax())                    # position in row order
    row, col = divmod(flat, img.shape[1])        # each row holds W pixels
    return row, col

**Why it works.** In `uint8`, `77 * img[..., 0]` keeps only the product modulo 256, and so
does the sum. Then `// 256` raises an `OverflowError`, since NumPy will not turn 256 into
a `uint8` (on NumPy 2 or later, and NumPy 1.x promoted the array to a wider integer instead),
and `/ 256` ranks the pixels by the wrapped sums, which put the dark blue pixel
(29 times 35 is 1015, stored as 247) above the bright one (51010, stored as 66). Converting to `int64` first
keeps every product exact. In row order, position $p$ of an $H \times W$ table sits in row
$p // W$ and column $p \bmod W$, because each row holds $W$ entries, which is what
`divmod(flat, W)` returns.